## Nota de trazabilidad (Parte II)

> Celda añadida en la Parte II del proyecto. **No se eliminó ni modificó ninguna celda del notebook original**; solo se agregaron celdas de documentación (esta, y las marcadas como *[Añadido en Parte II]* antes del cierre del navegador).

- **Entorno requerido.** Este notebook se diseñó para **Google Colab**: instala Google Chrome y Selenium con comandos de shell (`!wget`, `!apt-get`, `!pip`) y necesita acceso de red a `https://www.plazavea.com.pe`. Por ello **no se re-ejecuta** como parte del pipeline reproducible del repositorio; el pipeline de la Parte II parte del CSV ya extraído.
- **Dataset oficial.** El archivo `data/raw/productos_plazavea_raw.csv` (**1438 registros × 12 variables**, 1438 SKU únicos, sha256 `3a3833e8312f8d27566083b81cd02a86535055587ba38f25de4e5d05f4348f3a`) proviene de la corrida de extracción realizada en la **Parte I** y es la única fuente de datos utilizada en la Parte II. Se conserva sin modificaciones.
- **Outputs visibles en este notebook.** Las salidas que se observan a continuación corresponden a **una corrida posterior** del mismo código (1407 filas extraídas → 1397 SKU tras deduplicar; en esa corrida la página 16/17 tuvo un *timeout*). No son la corrida que generó el CSV oficial; se conservan únicamente como **evidencia del proceso de adquisición** (selectores, paginación, limpieza inicial).
- **Snapshot.** El contenido de plazavea.com.pe cambia continuamente (surtido, precios, promociones, orden del listado). Cada ejecución produce una **fotografía distinta** del catálogo; por eso las cifras de una nueva corrida no coincidirán con las del dataset oficial.
- **Hallazgo de calidad relevante.** En el CSV oficial la variable `discount` vale 0 en el 100% de los registros, aunque 479 productos tienen `current_price < regular_price`. Ver la sección *[Añadido en Parte II]* al final del notebook para el diagnóstico y la mejora propuesta.


# Adquisición y caracterización de precios y promociones de productos de Abarrotes de Plaza Vea Perú

## Objetivo general

Analizar qué marcas ofrecen los mayores descuentos dentro de las distintas categorías de Abarrotes de Plaza Vea, con el fin de identificar patrones promocionales y oportunidades de posicionamiento comercial.

### Integrantes

- Sebastian Loli Gonzalez 202420022
- Piero Max Ortiz Villafuerte 202410409
- Diego Mauricio Alarcon Ballivian 202410104
- Joaquin Justo Guillen 202410394
- Sofía Fridkas García Chaupis 202310531

## Decisión técnica y camino seguido

Al inicio se probó `requests` junto con `BeautifulSoup`. La página respondía correctamente, pero los productos aparecían principalmente como contenedores `g-producto` sin nombre, precio ni demás información comercial, debido a que Plaza Vea completa las tarjetas mediante JavaScript.

Por ello se utilizó **Selenium en modo headless** para cargar y renderizar la página. Luego se obtiene el HTML con `driver.page_source` y se analiza con **BeautifulSoup**, manteniendo las operaciones: `find()`, `find_all()`, `.text` y `.get()`.

Durante la exploración se identificó la tarjeta completa `ga-product-item`, que contiene información como SKU, marca, nombre, categoría, vendedor y stock, además de los elementos internos para presentación, precios, descuento, URL e imagen.

### Decisión sobre la paginación

Se probaron primero el scroll y los controles internos de paginación, pero no resultaron estables para automatizar el cambio de productos. Finalmente se comprobó que Plaza Vea permite acceder directamente a cada página mediante el parámetro:

`https://www.plazavea.com.pe/abarrotes?page=N`

La página 2 mostró productos distintos a la primera, por lo que esta forma se utilizará para recorrer varias páginas del catálogo.

## 1. Instalación y librerías

In [ ]:
# Instala Google Chrome y Selenium en Google Colab.
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt-get install -y ./google-chrome-stable_current_amd64.deb
!pip install -q -U selenium

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
Note, selecting 'google-chrome-stable' instead of './google-chrome-stable_current_amd64.deb'
The following additional packages will be installed:
  at-spi2-common at-spi2-core gsettings-desktop-schemas libatk-bridge2.0-0t64
  libatk1.0-0t64 libatspi2.0-0t64 libxcomposite1 libxtst6 session-migration
The following NEW packages will be installed:
  at-spi2-common at-spi2-core google-chrome-stable gsettings-desktop-schemas
  libatk-bridge2.0-0t64 libatk1.0-0t64 libatspi2.0-0t64 libxcomposite1
  libxtst6 session-migration
0 upgraded, 10 newly installed, 0 to remove and 0 not upgraded.
Need to get 331 kB/142 MB of archives.
After this operation, 456 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble/main amd64 at-spi2-common all 2.52.0-1build1 [8,674 B]
Get:2 http://archive.ubuntu.com/ubuntu noble/main amd64 libatk1.0-0t64 amd64 2.52.0-1build1 [55.3 kB]
Get:3 h

In [ ]:
# Importa las librerías necesarias.
from bs4 import BeautifulSoup as BSoup
import pandas as pd
import requests
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
import time

## 2. Configuración de Selenium

In [ ]:
# Configura Chrome para ejecutarse sin interfaz gráfica.
options = Options()
options.add_argument("--headless=new")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.binary_location = "/usr/bin/google-chrome"

driver = webdriver.Chrome(options=options)

## 3. Carga inicial de Plaza Vea

In [ ]:
# Abre la categoría Abarrotes.
url = "https://www.plazavea.com.pe/abarrotes"

driver.get(url)
time.sleep(5)

print(driver.title)

Abarrotes | plazaVea


In [ ]:
# Obtiene el HTML renderizado y crea el objeto BeautifulSoup.
html = driver.page_source

soup = BSoup(
    html,
    "html.parser"
)

## 4. Identificación de las tarjetas de producto

In [ ]:
# Busca las tarjetas completas renderizadas.
cardList = soup.find_all(
    "div",
    "ga-product-item"
)

print("Productos encontrados:", len(cardList))

Productos encontrados: 57


In [ ]:
# Revisa los atributos disponibles en la primera tarjeta.
card = cardList[0]

card.attrs

{'class': ['HA', 'Showcase', 'Showcase--food', 'ga-product-item'],
 'data-sku': '10725999',
 'data-unit-multiplier': '1',
 'data-has-attachment': 'false',
 'data-product-type': 'Supermercado',
 'data-stock': 'true',
 'data-ga-index': '1',
 'data-ga-price': '17.9',
 'data-ga-variant': 'food',
 'data-ga-category': 'Abarrotes / Salsas, Cremas y Condimentos / Mayonesa',
 'data-ga-seller': 'plaza vea',
 'data-ga-name': 'Mayonesa ALACENA Doypack 850g',
 'data-ga-brand': 'ALACENA',
 'data-is-weighable': 'false',
 'data-department-id': '431'}

In [ ]:
# Verifica algunos datos del primer producto.
print("SKU:", card.get("data-sku"))
print("Marca:", card.get("data-ga-brand"))
print("Nombre:", card.get("data-ga-name"))
print("Categoría:", card.get("data-ga-category"))
print("Precio:", card.get("data-ga-price"))

SKU: 10725999
Marca: ALACENA
Nombre: Mayonesa ALACENA Doypack 850g
Categoría: Abarrotes / Salsas, Cremas y Condimentos / Mayonesa
Precio: 17.9


## 5. Validación de la paginación

In [ ]:
# Abre directamente la página 2 mediante el parámetro page.
urlPagina2 = "https://www.plazavea.com.pe/abarrotes?page=2"

driver.get(urlPagina2)
time.sleep(5)

print(driver.current_url)

https://www.plazavea.com.pe/abarrotes?page=2


In [ ]:
# Comprueba que la página 2 contiene productos diferentes.
cardsPagina2 = driver.find_elements(
    By.CLASS_NAME,
    "ga-product-item"
)

print("Productos renderizados:", len(cardsPagina2))

for card in cardsPagina2[:10]:
    print(
        card.get_attribute("data-sku"),
        card.get_attribute("data-ga-name")
    )

Productos renderizados: 50
11289113 Pack Galletas Saladas RITZ Bolsa 20g Paquete 6un x 2un
11289110 Pack Galletas OREO Original Paquete 6un x 2
10947880 Galletas Saladas RITZ sabor Queso Paquete 6un
11187019 Pack Galletas OREO Original Paquete 12un x 2
11692 Galletas OREO Sabor Cookies & Cream Paquete 6un
2729 Galletas OREO Sabor Vainilla Paquete 6un
11308901 Pack Galletas Saladas RITZ Sabor a Queso Bolsa 30g Paquete 6un x 2un
1311 Galletas Saladas RITZ Mini Original Bolsa 50g
1450 Galletas OREO Sabor Chocolate Paquete 6un
12302304 Pack Galletas OREO Original Paquete 12un + Galletas RITZ Original Paquete 6un


La prueba anterior permite verificar que `?page=2` carga productos diferentes a los de la página inicial. Por ello, el scraping multipágina se realizará modificando directamente el valor de `page`.

## 6. Extracción multipágina

In [ ]:
# Recorre 30 páginas y acumula los productos encontrados.
productData = []

for pagina in range(1, 31):

    try:

        urlPagina = (
            "https://www.plazavea.com.pe/abarrotes"
            "?page=" + str(pagina)
        )

        driver.get(urlPagina)
        time.sleep(4)

        html = driver.page_source

        soup = BSoup(
            html,
            "html.parser"
        )

        cardList = soup.find_all(
            "div",
            "ga-product-item"
        )

        # Si no encuentra productos, intenta cargar nuevamente.
        if len(cardList) == 0:

            driver.refresh()
            time.sleep(5)

            html = driver.page_source

            soup = BSoup(
                html,
                "html.parser"
            )

            cardList = soup.find_all(
                "div",
                "ga-product-item"
            )

        print(
            "Página:",
            pagina,
            "- Productos encontrados:",
            len(cardList)
        )

        for card in cardList:

            elem = {}

            elem["sku"] = card.get("data-sku")
            elem["brand"] = card.get("data-ga-brand")
            elem["name"] = card.get("data-ga-name")
            elem["category"] = card.get("data-ga-category")
            elem["seller"] = card.get("data-ga-seller")
            elem["stock"] = card.get("data-stock")

            presentation = card.find(
                "div",
                "Showcase__units-reference"
            )

            if presentation:
                elem["presentation"] = presentation.text.strip()
            else:
                elem["presentation"] = None

            salePrice = card.find(
                "div",
                "Showcase__salePrice"
            )

            if salePrice:
                elem["current_price"] = salePrice.get("data-price")
            else:
                elem["current_price"] = None

            oldPrice = card.find(
                "div",
                "Showcase__oldPrice"
            )

            if oldPrice:
                elem["regular_price"] = oldPrice.text.strip()
            else:
                elem["regular_price"] = None

            discount = card.find(
                string=lambda text:
                text and text.strip().startswith("-")
                and "%" in text
            )

            if discount:
                elem["discount"] = discount.strip()
            else:
                elem["discount"] = None

            link = card.find(
                "a",
                "Showcase__link"
            )

            if link:
                elem["product_url"] = link.get("href")
            else:
                elem["product_url"] = None

            image = card.find(
                "img",
                "showcase__image"
            )

            if image:
                elem["image_url"] = image.get("src")
            else:
                elem["image_url"] = None

            productData.append(elem)

    except Exception as error:

        print(
            "Error en la página",
            pagina,
            ":",
            error
        )

Página: 1 - Productos encontrados: 57
Página: 2 - Productos encontrados: 50
Página: 3 - Productos encontrados: 50
Página: 4 - Productos encontrados: 50
Página: 5 - Productos encontrados: 50
Página: 6 - Productos encontrados: 50
Página: 7 - Productos encontrados: 50
Página: 8 - Productos encontrados: 50
Página: 9 - Productos encontrados: 50
Página: 10 - Productos encontrados: 50
Página: 11 - Productos encontrados: 50
Página: 12 - Productos encontrados: 50
Página: 13 - Productos encontrados: 50
Página: 14 - Productos encontrados: 50
Página: 15 - Productos encontrados: 50
Error en la página 16 : HTTPConnectionPool(host='localhost', port=44389): Read timed out. (read timeout=120)
Error en la página 17 : HTTPConnectionPool(host='localhost', port=44389): Read timed out. (read timeout=120)
Página: 18 - Productos encontrados: 50
Página: 19 - Productos encontrados: 50
Página: 20 - Productos encontrados: 50
Página: 21 - Productos encontrados: 50
Página: 22 - Productos encontrados: 50
Página: 23 

## 7. Construcción del dataset

In [ ]:
# Convierte los productos recolectados en un DataFrame.
dfProducts = pd.DataFrame(productData)

dfProducts.shape

(1407, 12)

In [ ]:
# Elimina productos repetidos usando el SKU.
dfProducts = dfProducts.drop_duplicates(
    subset="sku"
).reset_index(drop=True)

dfProducts.shape

(1397, 12)

In [ ]:
# Muestra una vista preliminar del dataset.
dfProducts.head()

,sku,brand,name,category,seller,stock,presentation,current_price,regular_price,discount,product_url,image_url
0,10614795,PRIMOR,Aceite Vegetal Premium PRIMOR Botella 900ml,Abarrotes / Aceite / Aceite Vegetal,plaza vea,true,Botella 900ml,9.20,S/ 11 x Und,-16%,https://www.plazavea.com.pe/aceite-vegetal-pre...,https://plazavea.vteximg.com.br/arquivos/ids/3...
1,12069688,BON O BON,Minis Bombones de Chocolate BON O BON Multipac...,Abarrotes / Chocolatería / Bombones,plaza vea,true,Bolsa 360g,26,None,None,https://www.plazavea.com.pe/minis-bombones-de-...,https://plazavea.vteximg.com.br/arquivos/ids/3...
2,10725999,ALACENA,Mayonesa ALACENA Doypack 850g,"Abarrotes / Salsas, Cremas y Condimentos / May...",plaza vea,true,Doypack 850g,17.90,S/ 19.60 x Und,-8%,https://www.plazavea.com.pe/mayonesa-alacena-d...,https://plazavea.vteximg.com.br/arquivos/ids/3...
3,5040,ALACENA,Crema de Ají Tarí ALACENA Doypack 400g,"Abarrotes / Salsas, Cremas y Condimentos / Ají...",plaza vea,true,Doypack 400g,12.30,S/ 13.50 x Und,-8%,https://www.plazavea.com.pe/crema-de-aji-tari-...,
4,12069682,AMBROSOLI,Gomas AMBROSOLI Cerebros Bolsa 250g,Abarrotes / Galletas y Golosinas / Gomitas y G...,plaza vea,true,Bolsa 250g,14.50,None,None,https://www.plazavea.com.pe/gomas-ambrosoli-ce...,


## 8. Limpieza de variables numéricas

In [ ]:
# Convierte el precio actual a tipo numérico.
dfProducts["current_price"] = pd.to_numeric(
    dfProducts["current_price"]
)

In [ ]:
# Limpia y convierte el precio regular.
dfProducts["regular_price"] = dfProducts["regular_price"].str.replace(
    "S/ ",
    "",
    regex=False
)

dfProducts["regular_price"] = dfProducts["regular_price"].str.replace(
    " x Und",
    "",
    regex=False
)

dfProducts["regular_price"] = pd.to_numeric(
    dfProducts["regular_price"]
)

# Si no hay promoción, se usa el precio actual como precio regular.
dfProducts["regular_price"] = dfProducts["regular_price"].fillna(
    dfProducts["current_price"]
)

In [ ]:
# Convierte el descuento a porcentaje numérico.
dfProducts["discount"] = dfProducts["discount"].str.extract(
    r"(-\d+)%",
    expand=False
)

dfProducts["discount"] = pd.to_numeric(
    dfProducts["discount"],
    errors="coerce"
)

dfProducts["discount"] = abs(
    dfProducts["discount"]
)

dfProducts["discount"] = dfProducts["discount"].fillna(0)

### Análisis del Dataset Preliminar

In [ ]:
print("Número de registros:", dfProducts.shape[0])
print("Número de atributos:", dfProducts.shape[1])

Número de registros: 1397
Número de atributos: 12


In [ ]:
dfProducts.dtypes

,0
sku,object
brand,object
name,object
category,object
seller,object
stock,object
presentation,object
current_price,float64
regular_price,float64
discount,float64


In [ ]:
dfProducts.describe()

,current_price,regular_price,discount
count,1397.000000,1397.000000,1397.000000
mean,10.374195,11.069241,5.634932
std,9.508592,10.031521,9.136023
min,0.800000,0.800000,0.000000
25%,4.700000,5.000000,0.000000
50%,7.600000,7.900000,0.000000
75%,12.900000,13.900000,11.000000
max,89.200000,105.000000,59.000000


In [ ]:
(dfProducts.isnull().mean()*100).round(2)

,0
sku,0.00
brand,0.00
name,0.00
category,0.00
seller,0.00
stock,0.00
presentation,0.07
current_price,0.00
regular_price,0.00
discount,0.00


In [ ]:
print("Marcas distintas:", dfProducts["brand"].nunique())
print("Categorías distintas:", dfProducts["category"].nunique())
print("Vendedores distintos:", dfProducts["seller"].nunique())

Marcas distintas: 257
Categorías distintas: 70
Vendedores distintos: 1


In [ ]:
dfProducts["category"].value_counts()

,count
category,
Abarrotes / Galletas y Golosinas / Galletas Dulces,117
Abarrotes / Galletas y Golosinas / Caramelos y Chupetes,71
Abarrotes / Chocolatería / Chocolates y Lentejas,60
Packs / Packs Abarrotes,58
Abarrotes / Conservas / Pescados en Conserva,49
...,...
Abarrotes / Snacks y Piqueos / Pretzels,2
Abarrotes / Snacks y Piqueos / Salsas y Dips,2
Abarrotes / Azúcar y Endulzantes / Azúcar en Polvo,2


In [ ]:
dfProducts["brand"].value_counts().head(10)

,count
brand,
BELL'S,180
COSTA,39
SIBARITA,33
ARCOR,33
DON VITTORIO,31
AMBROSOLI,28
VILLA NATURA,25
MOLITALIA,23
COSTEÑO,22


In [ ]:
print("Precio mínimo:", dfProducts["current_price"].min())
print("Precio máximo:", dfProducts["current_price"].max())
print("Precio promedio:", round(dfProducts["current_price"].mean(),2))

Precio mínimo: 0.8
Precio máximo: 89.2
Precio promedio: 10.37


In [ ]:
print(dfProducts[["current_price","regular_price","discount"]].describe())

       current_price  regular_price     discount
count    1397.000000    1397.000000  1397.000000
mean       10.374195      11.069241     5.634932
std         9.508592      10.031521     9.136023
min         0.800000       0.800000     0.000000
25%         4.700000       5.000000     0.000000
50%         7.600000       7.900000     0.000000
75%        12.900000      13.900000    11.000000
max        89.200000     105.000000    59.000000


## 9. Validación del dataset

In [ ]:
# Revisa la estructura y los tipos de datos.
dfProducts.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1397 entries, 0 to 1396
Data columns (total 12 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   sku            1397 non-null   object 
 1   brand          1397 non-null   object 
 2   name           1397 non-null   object 
 3   category       1397 non-null   object 
 4   seller         1397 non-null   object 
 5   stock          1397 non-null   object 
 6   presentation   1396 non-null   object 
 7   current_price  1397 non-null   float64
 8   regular_price  1397 non-null   float64
 9   discount       1397 non-null   float64
 10  product_url    1397 non-null   object 
 11  image_url      1397 non-null   object 
dtypes: float64(3), object(9)
memory usage: 131.1+ KB


In [ ]:
# Revisa la cantidad de valores nulos por columna.
dfProducts.isnull().sum()

,0
sku,0
brand,0
name,0
category,0
seller,0
stock,0
presentation,1
current_price,0
regular_price,0
discount,0


In [ ]:
# Verifica que no existan SKU repetidos.
dfProducts["sku"].duplicated().sum()

np.int64(0)

In [ ]:
# Revisa el tamaño final obtenido.
dfProducts.shape

(1397, 12)

In [ ]:
# Verifica duplicados por SKU.
dfProducts["sku"].duplicated().sum()

np.int64(0)

In [ ]:
# Verifica valores nulos por columna.
dfProducts.isnull().sum()

,0
sku,0
brand,0
name,0
category,0
seller,0
stock,0
presentation,1
current_price,0
regular_price,0
discount,0


In [ ]:
# Revisa tipos de datos y cantidad de registros.
dfProducts.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1397 entries, 0 to 1396
Data columns (total 12 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   sku            1397 non-null   object 
 1   brand          1397 non-null   object 
 2   name           1397 non-null   object 
 3   category       1397 non-null   object 
 4   seller         1397 non-null   object 
 5   stock          1397 non-null   object 
 6   presentation   1396 non-null   object 
 7   current_price  1397 non-null   float64
 8   regular_price  1397 non-null   float64
 9   discount       1397 non-null   float64
 10  product_url    1397 non-null   object 
 11  image_url      1397 non-null   object 
dtypes: float64(3), object(9)
memory usage: 131.1+ KB


## Diccionario de datos

| Variable | Tipo | Descripción |
|---|---|---|
| `sku` | Categórica | Identificador único del producto en Plaza Vea. |
| `brand` | Categórica | Marca comercial del producto. |
| `name` | Categórica | Nombre comercial del producto. |
| `category` | Categórica | Categoría y subcategoría a la que pertenece el producto. |
| `seller` | Categórica | Vendedor encargado de comercializar el producto. |
| `stock` | Binaria / Categórica | Indica si el producto se encuentra disponible en stock. |
| `presentation` | Categórica | Presentación comercial del producto, por ejemplo paquete, bolsa, botella o número de unidades. |
| `current_price` | Numérica | Precio actual de venta del producto, expresado en soles. |
| `regular_price` | Numérica | Precio regular del producto antes de aplicar una promoción o descuento. |
| `discount` | Numérica | Porcentaje de descuento aplicado al producto. El valor `0` indica que no existe promoción. |
| `product_url` | Categórica | URL de la página individual del producto en Plaza Vea. |
| `image_url` | Categórica | URL correspondiente a la imagen principal del producto. |

In [ ]:
# Exporta el dataset final en formato CSV UTF-8.
dfProducts.to_csv(
    "productos_plazavea.csv",
    index=False,
    encoding="utf-8"
)

In [ ]:
dfProducts.shape

(1397, 12)

In [ ]:
for col in dfProducts.columns:
    if dfProducts[col].dtype == "object":
        print(col, "→ tamaño máx:", dfProducts[col].astype(str).str.len().max())
    else:
        print(col, "→ rango:", dfProducts[col].min(), "a", dfProducts[col].max())

sku → tamaño máx: 8
brand → tamaño máx: 25
name → tamaño máx: 116
category → tamaño máx: 72
seller → tamaño máx: 9
stock → tamaño máx: 4
presentation → tamaño máx: 23
current_price → rango: 0.8 a 89.2
regular_price → rango: 0.8 a 105.0
discount → rango: 0.0 a 59.0
product_url → tamaño máx: 141
image_url → tamaño máx: 98


## 9b. Exportación del dataset concatenado previo a la deduplicación *[Añadido en Parte II]*

Para que futuras corridas sean trazables, se recomienda guardar el `DataFrame` **tal como sale del scraping** (concatenación de todas las páginas, antes de `drop_duplicates` y antes de convertir precios y descuento). Así se conserva el texto crudo de cada campo (por ejemplo, el texto original del descuento) y se puede auditar cualquier transformación posterior.

La celda siguiente **no altera la lógica original**: reconstruye el `DataFrame` desde la lista `productData` y lo exporta. Para no sobrescribir el snapshot oficial de la Parte I, si `../data/raw/productos_plazavea_raw.csv` ya existe se escribe un archivo con marca de tiempo.


In [ ]:
# [Añadido en Parte II] Finalidad: exportar el DataFrame concatenado de todas las páginas
# ANTES de la deduplicación por SKU y ANTES de la limpieza numérica, para trazabilidad
# de futuras corridas. No modifica dfProducts ni la lógica original del notebook.
import os
from datetime import datetime

os.makedirs("../data/raw", exist_ok=True)

dfRawConcat = pd.DataFrame(productData)   # datos crudos, tal como se extrajeron

rawPath = "../data/raw/productos_plazavea_raw.csv"
if os.path.exists(rawPath):
    # Protege el snapshot oficial de la Parte I (1438 x 12): no se sobrescribe.
    rawPath = (
        "../data/raw/productos_plazavea_raw_"
        + datetime.now().strftime("%Y%m%d_%H%M%S")
        + ".csv"
    )

dfRawConcat.to_csv(rawPath, index=False, encoding="utf-8")
print("Exportado:", rawPath, "- forma:", dfRawConcat.shape)


## 9c. Diagnóstico: pérdida del descuento en la corrida oficial *[Añadido en Parte II]*

**Hallazgo.** En el CSV oficial (`data/raw/productos_plazavea_raw.csv`, corrida de la Parte I) la columna `discount` es **0.0 en el 100% de los registros** (mínimo = máximo = 0), mientras que **479 productos** presentan `current_price < regular_price`. El informe de la Parte I reportaba un rango de 0 a 59%, que corresponde a otra corrida (la de los outputs visibles en este notebook), no al CSV entregado.

**Causa probable.** El descuento se obtiene en dos pasos frágiles:
1. En la extracción se busca un nodo de texto que **empiece con "-" y contenga "%"** (`card.find(string=lambda text: text.strip().startswith("-") and "%" in text)`). Si el sitio muestra el porcentaje con otro formato (sin el signo "-", con espacios, en un nodo distinto o cargado de forma diferida), la búsqueda devuelve `None`.
2. En la limpieza se extrae el patrón `(-\d+)%` y luego se aplica **`fillna(0)`**. Cualquier valor no encontrado o no parseable se convierte silenciosamente en 0, por lo que un fallo de captura queda indistinguible de "sin promoción".

**Consecuencia para la Parte II.** La columna `discount` almacenada se considera **inválida**. En el preprocesamiento se reconstruye el descuento a partir de los precios: `discount_pct = (regular_price − current_price) / regular_price × 100`, y se documenta en el diccionario final.

### Mejora propuesta (no aplicada; la lógica original se mantiene)

- Guardar el **texto crudo del descuento** en una columna propia (p. ej. `elem["discount_raw"] = discount.strip() if discount else None`) **antes** de convertirlo a número.
- Convertir a número en una columna separada (`discount`) **sin `fillna(0)`**: dejar `NaN` cuando no se capture, y contar cuántos `NaN` hay.
- Añadir una validación de consistencia tras la extracción: si `current_price < regular_price` y el descuento es `NaN`/0, registrar una alerta (el selector probablemente falló).
- Usar un patrón más tolerante, p. ej. `r"-?\s*(\d+)\s*%"`, y verificarlo sobre una muestra de tarjetas.


## 10. Cierre del navegador

In [ ]:
# Cierra la sesión de Selenium cuando termina la extracción.
driver.quit()